In [1]:
import urllib.request
import gzip
import os

downloads = os.path.expanduser('~/Downloads')
os.chdir(downloads)

print("Downloading BRCA expression...")
urllib.request.urlretrieve(
    'https://tcga.xenahubs.net/download/TCGA.BRCA.sampleMap/HiSeqV2.gz',
    'brca_expression.tsv.gz'
)

print("Extracting BRCA...")
with gzip.open('brca_expression.tsv.gz', 'rb') as f:
    open('brca_expression.tsv', 'wb').write(f.read())

print("Downloading SKCM expression...")
urllib.request.urlretrieve(
    'https://tcga.xenahubs.net/download/TCGA.SKCM.sampleMap/HiSeqV2.gz',
    'skcm_expression.tsv.gz'
)

print("Extracting SKCM...")
with gzip.open('skcm_expression.tsv.gz', 'rb') as f:
    open('skcm_expression.tsv', 'wb').write(f.read())



print("\n✓ BRCA and SKCM expression data downloaded and extracted")

Extracting BRCA...
Extracting SKCM...

✓ BRCA and SKCM expression data downloaded and extracted


In [2]:
import pandas as pd
import os

downloads = os.path.expanduser('~/Downloads')
os.chdir(downloads)

print("Loading expression data...")
brca = pd.read_csv('brca_expression.tsv', sep='\t', index_col=0)
skcm = pd.read_csv('skcm_expression.tsv', sep='\t', index_col=0)

print(f"BRCA shape: {brca.shape} (genes × samples)")
print(f"SKCM shape: {skcm.shape} (genes × samples)")

# Combine
combined = pd.concat([brca, skcm], axis=1)
print(f"\nCombined shape: {combined.shape}")

# Create metadata with cancer type labels
sample_metadata = pd.DataFrame(index=combined.columns)
sample_metadata['sample_barcode'] = combined.columns

# Assign cancer type based on TCGA barcode prefix
def get_cancer_type(barcode):
    if 'TCGA-A' in barcode or 'TCGA-B' in barcode or 'TCGA-D' in barcode:
        return 'BRCA'
    else:
        return 'SKCM'

sample_metadata['cancer_type'] = [get_cancer_type(s) for s in combined.columns]

print(f"\nCancer type distribution:")
print(sample_metadata['cancer_type'].value_counts())

# Transpose expression (rows=samples, columns=genes)
expr_t = combined.T
expr_t['sample_barcode'] = expr_t.index

# Merge
merged = expr_t.merge(sample_metadata[['sample_barcode', 'cancer_type']], 
                       on='sample_barcode', how='left')

print(f"\nMerged data shape: {merged.shape}")
print(f"Merged has {merged.shape[0]} samples and {merged.shape[1]-1} genes + metadata")

# Save
merged.to_csv('tcga_brca_skcm_merged.csv', index=False)
print(f"\n✓ Saved: tcga_brca_skcm_merged.csv")

Loading expression data...
BRCA shape: (20530, 1218) (genes × samples)
SKCM shape: (20530, 474) (genes × samples)

Combined shape: (20530, 1692)

Cancer type distribution:
cancer_type
BRCA    969
SKCM    723
Name: count, dtype: int64

Merged data shape: (1692, 20532)
Merged has 1692 samples and 20531 genes + metadata

✓ Saved: tcga_brca_skcm_merged.csv


In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

data = pd.read_csv('tcga_brca_skcm_merged.csv')

print("PREPROCESSING")
print("="*50)

# Separate metadata and expression
metadata = data[['sample_barcode', 'cancer_type']].copy()
expr = data.drop(['sample_barcode', 'cancer_type'], axis=1)

print(f"Data shape: {expr.shape} (samples × genes)")
print(f"Cancer type distribution:")
print(metadata['cancer_type'].value_counts())

# Step 1: Handle missing values (simple)
print(f"\n1. Checking for missing values...")
missing_count = expr.isna().sum().sum()
print(f"   Missing values: {missing_count}")
if missing_count > 0:
    expr = expr.fillna(expr.mean())
    print(f"   Filled with mean values")
else:
    print(f"   No missing values ✓")

# Data is ready
print(f"\nPreprocessed shape: {expr.shape}")

# Combine and save
preprocessed = pd.concat([metadata, expr], axis=1)
preprocessed.to_csv('tcga_brca_skcm_preprocessed.csv', index=False)
print(f"\n✓ Saved: tcga_brca_skcm_preprocessed.csv")

PREPROCESSING
Data shape: (1692, 20530) (samples × genes)
Cancer type distribution:
cancer_type
BRCA    969
SKCM    723
Name: count, dtype: int64

1. Checking for missing values...
   Missing values: 0
   No missing values ✓

Preprocessed shape: (1692, 20530)

✓ Saved: tcga_brca_skcm_preprocessed.csv


In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pickle
import os

os.chdir(os.path.expanduser('~/Downloads'))

data = pd.read_csv('tcga_brca_skcm_preprocessed.csv')

print("TRAIN/TEST SPLIT & SCALING")
print("="*50)

# Separate features and target
X = data.drop(['sample_barcode', 'cancer_type'], axis=1)
y = data['cancer_type']  # TARGET: BRCA or SKCM

print(f"Features (genes): {X.shape[1]}")
print(f"Samples: {X.shape[0]}")
print(f"\nTarget distribution:")
print(y.value_counts())
print()

# Train/test split (80/20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"\nTrain distribution:")
print(y_train.value_counts())
print(f"\nTest distribution:")
print(y_test.value_counts())
print()

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Scaling complete")
print(f"Train: mean={X_train_scaled.mean():.4f}, std={X_train_scaled.std():.4f}")
print(f"Test: mean={X_test_scaled.mean():.4f}, std={X_test_scaled.std():.4f}")

# Save
np.save('X_train_scaled.npy', X_train_scaled)
np.save('X_test_scaled.npy', X_test_scaled)
np.save('y_train.npy', y_train.values)
np.save('y_test.npy', y_test.values)

with open('gene_names.pkl', 'wb') as f:
    pickle.dump(X.columns.tolist(), f)

print(f"\n✓ Saved: X_train_scaled.npy, X_test_scaled.npy, y_train.npy, y_test.npy, gene_names.pkl")

TRAIN/TEST SPLIT & SCALING
Features (genes): 20530
Samples: 1692

Target distribution:
cancer_type
BRCA    969
SKCM    723
Name: count, dtype: int64

Train set: 1353 samples
Test set: 339 samples

Train distribution:
cancer_type
BRCA    775
SKCM    578
Name: count, dtype: int64

Test distribution:
cancer_type
BRCA    194
SKCM    145
Name: count, dtype: int64

Scaling complete
Train: mean=0.0000, std=0.9936
Test: mean=0.0108, std=0.9991

✓ Saved: X_train_scaled.npy, X_test_scaled.npy, y_train.npy, y_test.npy, gene_names.pkl


In [5]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import pickle
import os

os.chdir(os.path.expanduser('~/Downloads'))

# Load scaled data
X_train = np.load('X_train_scaled.npy')
X_test = np.load('X_test_scaled.npy')
y_train = np.load('y_train.npy', allow_pickle=True)
y_test = np.load('y_test.npy', allow_pickle=True)

print("TRAINING 3 MODELS — CANCER TYPE CLASSIFICATION")
print("="*50)

models = {}
results = {}

# 1. Logistic Regression
print("\n1. Logistic Regression...")
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

acc_lr = accuracy_score(y_test, y_pred_lr)
f1_lr = f1_score(y_test, y_pred_lr, average='weighted', zero_division=0)

results['Logistic Regression'] = {
    'accuracy': acc_lr,
    'f1': f1_lr,
    'predictions': y_pred_lr
}
models['Logistic Regression'] = lr
print(f"   ✓ Accuracy: {acc_lr:.4f}")

# 2. Random Forest
print("\n2. Random Forest...")
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

acc_rf = accuracy_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf, average='weighted', zero_division=0)

results['Random Forest'] = {
    'accuracy': acc_rf,
    'f1': f1_rf,
    'predictions': y_pred_rf
}
models['Random Forest'] = rf
print(f"   ✓ Accuracy: {acc_rf:.4f}")

# 3. SVM
print("\n3. SVM...")
svm = SVC(kernel='rbf', random_state=42)
svm.fit(X_train, y_train)
y_pred_svm = svm.predict(X_test)

acc_svm = accuracy_score(y_test, y_pred_svm)
f1_svm = f1_score(y_test, y_pred_svm, average='weighted', zero_division=0)

results['SVM'] = {
    'accuracy': acc_svm,
    'f1': f1_svm,
    'predictions': y_pred_svm
}
models['SVM'] = svm
print(f"   ✓ Accuracy: {acc_svm:.4f}")

# Summary
print("\n" + "="*50)
print("PERFORMANCE SUMMARY")
print("="*50)
for model in results:
    print(f"{model:20} | Accuracy: {results[model]['accuracy']:.4f} | F1: {results[model]['f1']:.4f}")

# Confusion matrices
print("\n" + "="*50)
print("CONFUSION MATRICES & CLASSIFICATION REPORTS")
print("="*50)

for model_name in ['Logistic Regression', 'Random Forest', 'SVM']:
    y_pred = results[model_name]['predictions']
    cm = confusion_matrix(y_test, y_pred)
    print(f"\n{model_name}:")
    print(cm)
    print(classification_report(y_test, y_pred))

# Save models
with open('trained_models.pkl', 'wb') as f:
    pickle.dump(models, f)
with open('model_results.pkl', 'wb') as f:
    pickle.dump(results, f)

print(f"\n✓ Saved: trained_models.pkl, model_results.pkl")

TRAINING 3 MODELS — CANCER TYPE CLASSIFICATION

1. Logistic Regression...
   ✓ Accuracy: 0.7581

2. Random Forest...
   ✓ Accuracy: 0.7227

3. SVM...
   ✓ Accuracy: 0.7168

PERFORMANCE SUMMARY
Logistic Regression  | Accuracy: 0.7581 | F1: 0.7579
Random Forest        | Accuracy: 0.7227 | F1: 0.7167
SVM                  | Accuracy: 0.7168 | F1: 0.7118

CONFUSION MATRICES & CLASSIFICATION REPORTS

Logistic Regression:
[[154  40]
 [ 42 103]]
              precision    recall  f1-score   support

        BRCA       0.79      0.79      0.79       194
        SKCM       0.72      0.71      0.72       145

    accuracy                           0.76       339
   macro avg       0.75      0.75      0.75       339
weighted avg       0.76      0.76      0.76       339


Random Forest:
[[162  32]
 [ 62  83]]
              precision    recall  f1-score   support

        BRCA       0.72      0.84      0.78       194
        SKCM       0.72      0.57      0.64       145

    accuracy                

In [6]:
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix
import pickle
import os

os.chdir(os.path.expanduser('~/Downloads'))

# Load models and data
with open('trained_models.pkl', 'rb') as f:
    models = pickle.load(f)

with open('gene_names.pkl', 'rb') as f:
    gene_names = pickle.load(f)

print("FEATURE IMPORTANCE ANALYSIS")
print("="*50)

# Random Forest feature importance
print("\nRandom Forest — Top 20 genes:")
print("-"*40)
rf = models['Random Forest']
importance_rf = rf.feature_importances_
top_genes_rf = np.argsort(importance_rf)[-20:][::-1]

for i, idx in enumerate(top_genes_rf, 1):
    print(f"{i:2d}. {gene_names[idx]:20} | Importance: {importance_rf[idx]:.6f}")

# Logistic Regression feature importance
print("\n\nLogistic Regression — Top 20 genes:")
print("-"*40)
lr = models['Logistic Regression']
coef_magnitudes = np.abs(lr.coef_[0])
top_genes_lr = np.argsort(coef_magnitudes)[-20:][::-1]

for i, idx in enumerate(top_genes_lr, 1):
    print(f"{i:2d}. {gene_names[idx]:20} | Coeff Mag: {coef_magnitudes[idx]:.6f}")

# Consensus genes
top20_rf_set = set([gene_names[i] for i in top_genes_rf])
top20_lr_set = set([gene_names[i] for i in top_genes_lr])
consensus = sorted(top20_rf_set & top20_lr_set)

print("\n" + "="*50)
print(f"CONSENSUS GENES ({len(consensus)} genes appear in top 20 of both models)")
print("="*50)
for gene in consensus:
    print(f"  • {gene}")

# Save
consensus_df = pd.DataFrame({'consensus_genes': consensus})
consensus_df.to_csv('consensus_genes.csv', index=False)

print(f"\n✓ Saved: consensus_genes.csv")

FEATURE IMPORTANCE ANALYSIS

Random Forest — Top 20 genes:
----------------------------------------
 1. TNFRSF14             | Importance: 0.003008
 2. KIAA0415             | Importance: 0.002924
 3. C19orf46             | Importance: 0.002849
 4. C10orf118            | Importance: 0.002602
 5. KIAA1671             | Importance: 0.002521
 6. CAPN13               | Importance: 0.002510
 7. KIAA1217             | Importance: 0.002497
 8. PBX1                 | Importance: 0.002393
 9. FLJ45983             | Importance: 0.002329
10. PTPRF                | Importance: 0.002186
11. TAZ                  | Importance: 0.002119
12. MAP7                 | Importance: 0.001729
13. CAPN10               | Importance: 0.001688
14. DUS3L                | Importance: 0.001660
15. MICALL2              | Importance: 0.001592
16. SDC4                 | Importance: 0.001514
17. MIB2                 | Importance: 0.001483
18. PARD6B               | Importance: 0.001447
19. DGCR14               | Importanc